<a href="https://colab.research.google.com/github/alwaysalearner1234/ML03/blob/main/human_action_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Checkout the code**

In [21]:
!git clone https://github.com/spmallick/learnopencv.git

Cloning into 'learnopencv'...
remote: Enumerating objects: 22309, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 22309 (delta 36), reused 28 (delta 24), pack-reused 22248 (from 2)
Receiving objects: 100% (22309/22309), 3.63 GiB | 17.14 MiB/s, done.
Resolving deltas: 100% (6836/6836), done.
Updating files: 100% (14160/14160), done.


In [22]:
%cd learnopencv/Human-Action-Recognition-Using-Detectron2-And-Lstm

/content/learnopencv/Human-Action-Recognition-Using-Detectron2-And-Lstm/learnopencv/Human-Action-Recognition-Using-Detectron2-And-Lstm/learnopencv/Human-Action-Recognition-Using-Detectron2-And-Lstm


**Install dependencies**

In [23]:
!pip install --upgrade pip setuptools wheel
!pip install pytorch-lightning
!pip install numpy
!pip install -r requirements.txt
!pip install torchmetrics

  Using cached Flask-1.1.2-py2.py3-none-any.whl.metadata (4.6 kB)
  Using cached Flask-Bootstrap-3.3.7.1.tar.gz (456 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached Flask-Uploads-0.2.1.tar.gz (7.6 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-1.19.5.zip (7.3 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
ERROR: Exception:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 107, in _run_wrapper
    status = _inner_run()
             ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 98, in _inner_run
    return self.run(options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dis

**Install Detectron2**

In [24]:
import torch
TORCH_VERSION = ".".join(torch.__version__.split(".")[:2])

if torch.cuda.is_available():
    CUDA_VERSION = torch.version.cuda.replace(".", "")
    print(f"Detected PyTorch version: {TORCH_VERSION}, CUDA version: {CUDA_VERSION}")
    # Install detectron2 that matches the current PyTorch and CUDA versions
    # See https://detectron2.readthedocs.io/tutorials/install.html for more options
    !pip install detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu{CUDA_VERSION}/torch{TORCH_VERSION}/index.html
else:
    print("CUDA is not available. Please change your Colab runtime to GPU (Runtime -> Change runtime type -> GPU).")
    print("After changing the runtime, please restart and run all cells from the beginning.")

Detected PyTorch version: 2.9, CUDA version: 126
Looking in links: https://dl.fbaipublicfiles.com/detectron2/wheels/cu126/torch2.9/index.html
ERROR: Could not find a version that satisfies the requirement detectron2 (from versions: none)
ERROR: No matching distribution found for detectron2


**Install ngrok for tunneling to the web application we are about to run on colab**

In [25]:
# Download ngrok for tunneling.
!if [ ! -f ./ngrok ]; then \
 wget https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip; \
 unzip -o ngrok-stable-linux-amd64.zip; \
 fi

--2026-02-09 13:44:45--  https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
Resolving bin.equinox.io (bin.equinox.io)... 35.71.179.82, 99.83.220.108, 75.2.60.68, ...
Connecting to bin.equinox.io (bin.equinox.io)|35.71.179.82|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13921656 (13M) [application/octet-stream]
Saving to: ‘ngrok-stable-linux-amd64.zip’

ngrok-stable-linux- 100%[===================>]  13.28M  4.48MB/s    in 3.0s    

2026-02-09 13:44:49 (4.48 MB/s) - ‘ngrok-stable-linux-amd64.zip’ saved [13921656/13921656]

Archive:  ngrok-stable-linux-amd64.zip
  inflating: ngrok                   


In [26]:
# Then start the web application.
port = 5000
!pkill ngrok || true # Suppress error if ngrok is not running
!pkill -f "python app.py" || true # Kill previous instances of app.py

# Start app.py in the background.
get_ipython().system_raw('python app.py &')

# And, forward the port using ngrok.
get_ipython().system_raw('./ngrok http {} &'.format(port))

^C


**Copy the app url generated from the below step**

In [27]:
# Get the public address from localhost:4040 (ngrok's web interface).
import time, urllib.request, json
# Give ngrok more time to startup and establish the tunnel.
# It might take a few seconds for the tunnel to become active.
for _ in range(10): # Try for up to 10 seconds
    try:
        time.sleep(1)
        ngrok_data = json.load(
            urllib.request.urlopen('http://localhost:4040/api/tunnels'))
        if ngrok_data['tunnels']:
            print(ngrok_data['tunnels'][0]['public_url'])
            break
    except urllib.error.URLError:
        continue
else:
    print("Ngrok tunnel not established after 10 seconds.")

Ngrok tunnel not established after 10 seconds.


**Run the application**

In [28]:
!python app.py

Traceback (most recent call last):
  File "/content/learnopencv/Human-Action-Recognition-Using-Detectron2-And-Lstm/learnopencv/Human-Action-Recognition-Using-Detectron2-And-Lstm/learnopencv/Human-Action-Recognition-Using-Detectron2-And-Lstm/app.py", line 13, in <module>
    from detectron2 import model_zoo
ModuleNotFoundError: No module named 'detectron2'


In [29]:
import torch
print(torch.__version__)
print(torch.version.cuda)


2.9.0+cu126
12.6


In [30]:
pip install detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu126/torch2.9/index.html


Looking in links: https://dl.fbaipublicfiles.com/detectron2/wheels/cu126/torch2.9/index.html
ERROR: Could not find a version that satisfies the requirement detectron2 (from versions: none)
ERROR: No matching distribution found for detectron2


In [31]:
pip install Werkzeug==2.3.8


In [32]:
cd learnopencv/Human-Action-Recognition-Using-Detectron2-And-Lstm


[Errno 2] No such file or directory: 'learnopencv/Human-Action-Recognition-Using-Detectron2-And-Lstm'
/content/learnopencv/Human-Action-Recognition-Using-Detectron2-And-Lstm/learnopencv/Human-Action-Recognition-Using-Detectron2-And-Lstm/learnopencv/Human-Action-Recognition-Using-Detectron2-And-Lstm


**Open the app url on browser to access the app**